Referências: https://andersonuyekita.github.io/notebooks/blog/2019/03/21/como-usar-o-gridsearchcv/
https://www.geeksforgeeks.org/machine-learning/k-nearest-neighbours/

In [1]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
from sklearn.model_selection import train_test_split, KFold, cross_val_score
from sklearn.neighbors import KNeighborsClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, ConfusionMatrixDisplay

df = pd.read_csv("dataset.csv")

In [4]:
from sklearn.model_selection import GridSearchCV

X = df.iloc[:, 1:17]
y = df.iloc[:, -1]
X_treino, X_teste, y_treino, y_teste = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

# KNN usa distância, então normalizamos (o scaler dentro do Pipeline é ajustado só com o treino de cada divisão)
modelo = Pipeline([
    ('scaler', StandardScaler()),
    ('knn', KNeighborsClassifier())
])

In [6]:
from sklearn.model_selection import StratifiedShuffleSplit

# Usando o constructor para criar o objeto sss
sss = StratifiedShuffleSplit(n_splits = 5,       # 5 simulações.
                             test_size = 0.2,    # 20% do dataset será de testes.
                             random_state = 42)  # Permitir a reprodutibilidade.

In [8]:
# range de parametros que o grid search vai testar
parametros = {'knn__n_neighbors': [5, 11, 21, 31, 51, 75, 101, 151, 201],
              'knn__weights': ['uniform', 'distance'],
              'knn__metric': ['euclidean', 'manhattan', 'chebyshev']
             }


# Criação do objeto do GridSearchCV.
grid = GridSearchCV(estimator = modelo,          # Pipeline (scaler + KNN).
                    param_grid = parametros,     # É aquele dicionário com valores para serem testados.
                    scoring = 'f1',              # Mesmo score dos outros GridSearch.
                    cv = sss,                    # Quantas vezes o dataset passa por cross validation (nosso K)
                    n_jobs = -1)                 # Usa todos os núcleos do processador.

In [14]:
# Treinando o grid.
grid.fit(X, y)

# Imprimindo os resultados.
pd.DataFrame(grid.cv_results_)  

pd.DataFrame(grid.cv_results_)[['params','rank_test_score','mean_test_score']]

,params,rank_test_score,mean_test_score
0,"{'knn__metric': 'euclidean', 'knn__n_neighbors...",51,0.811229
1,"{'knn__metric': 'euclidean', 'knn__n_neighbors...",51,0.811229
2,"{'knn__metric': 'euclidean', 'knn__n_neighbors...",40,0.831738
3,"{'knn__metric': 'euclidean', 'knn__n_neighbors...",41,0.831688
4,"{'knn__metric': 'euclidean', 'knn__n_neighbors...",28,0.845218
5,"{'knn__metric': 'euclidean', 'knn__n_neighbors...",27,0.845392
6,"{'knn__metric': 'euclidean', 'knn__n_neighbors...",26,0.848830
7,"{'knn__metric': 'euclidean', 'knn__n_neighbors...",25,0.849487
8,"{'knn__metric': 'euclidean', 'knn__n_neighbors...",20,0.856290
9,"{'knn__metric': 'euclidean', 'knn__n_neighbors...",19,0.857209


In [15]:
# Imprime os parâmetros que produziram o ".best_score_".
print(grid.best_params_)

# Imprimindo o score.
print(grid.best_score_)

{'knn__metric': 'manhattan', 'knn__n_neighbors': 201, 'knn__weights': 'distance'}
0.8722937636470902
